# CampusPulse — Ensemble Learning

This notebook compares individual classifiers with **Hard Voting, Soft Voting, Bagging and Boosting (XGBoost)**.

### Important
- Uses the **canonical processed dataset**: `data/processed/campuspulse_akgec_processed.csv`
- Does **not** repeat feature engineering.
- Reuses the saved classification preprocessor and PCA artifacts.
- Logistic Regression uses the saved PCA representation, matching the final classification pipeline.
- Tree/XGBoost models use the dense preprocessed representation.


In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

RANDOM_STATE = 42

# Robust project-root detection
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "campuspulse_akgec_processed.csv"
PREPROCESSOR_PATH = PROJECT_ROOT / "ml" / "models" / "classification" / "campuspulse_preprocessor.pkl"
PCA_PATH = PROJECT_ROOT / "ml" / "models" / "classification" / "campuspulse_pca.pkl"
FINAL_MODEL_PATH = PROJECT_ROOT / "ml" / "models" / "classification" / "final_priority_model.pkl"
METADATA_PATH = PROJECT_ROOT / "ml" / "models" / "classification" / "model_metadata.pkl"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)


Project root: c:\Users\acer\Documents\FinalTask\CampusPulse
Dataset: c:\Users\acer\Documents\FinalTask\CampusPulse\data\processed\campuspulse_akgec_processed.csv


In [2]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nPriority distribution:")
print(df["priority"].value_counts())


Dataset shape: (200000, 41)

Priority distribution:
priority
Medium      70197
Low         68808
High        42242
Critical    18753
Name: count, dtype: int64


In [3]:
# The processed dataset already contains the 40 classification features + target.
X = df.drop(columns="priority").copy()
y = df["priority"].copy()

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y,)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


Train: (160000, 40)
Test : (40000, 40)


In [4]:
preprocessor = joblib.load(PREPROCESSOR_PATH)
pca = joblib.load(PCA_PATH)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# PCA requires dense input.
X_train_dense = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed
X_test_dense = X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed

X_train_pca = pca.transform(X_train_dense)
X_test_pca = pca.transform(X_test_dense)

print("Processed:", X_train_dense.shape, X_test_dense.shape)
print("PCA     :", X_train_pca.shape, X_test_pca.shape)


Processed: (160000, 304) (40000, 304)
PCA     : (160000, 152) (40000, 152)


In [5]:
# Encode target labels once for voting/XGBoost.
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

print("Classes:", label_encoder.classes_)


Classes: ['Critical' 'High' 'Low' 'Medium']


## Individual Classifiers

The final saved Logistic Regression model is loaded from the classification pipeline. Decision Tree and Random Forest are trained as ensemble base learners. XGBoost represents the boosting approach.


In [6]:
final_lr = joblib.load(FINAL_MODEL_PATH)
metadata = joblib.load(METADATA_PATH)

if metadata.get("model") != "Logistic Regression":
    print("Warning: saved final model is", metadata.get("model"))

lr_model = final_lr

lr_pred_raw = lr_model.predict(X_test_pca)
lr_pred = label_encoder.transform(lr_pred_raw)

lr_proba = lr_model.predict_proba(X_test_pca)

print("Logistic Regression completed.")
print("LR predicted labels:", np.unique(lr_pred_raw))
print("Encoded labels:", np.unique(lr_pred))

Logistic Regression completed.
LR predicted labels: ['Critical' 'High' 'Low' 'Medium']
Encoded labels: [0 1 2 3]


In [ ]:
dt_model = DecisionTreeClassifier(max_depth=15,random_state=RANDOM_STATE,)
dt_model.fit(X_train_dense, y_train_enc)

dt_pred = dt_model.predict(X_test_dense)
dt_proba = dt_model.predict_proba(X_test_dense)

print("Decision Tree completed.")


Decision Tree completed.


In [8]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_model.fit(X_train_dense, y_train_enc)

rf_pred = rf_model.predict(X_test_dense)
rf_proba = rf_model.predict_proba(X_test_dense)

print("Random Forest completed.")


Random Forest completed.


In [9]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_state=RANDOM_STATE,
    eval_metric="mlogloss",
)

xgb_model.fit(X_train_dense, y_train_enc)

xgb_pred = xgb_model.predict(X_test_dense)
xgb_proba = xgb_model.predict_proba(X_test_dense)

print("XGBoost completed.")


XGBoost completed.


In [10]:
def metrics_row(name, y_true, pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, pred, average="weighted", zero_division=0),
        "F1": f1_score(y_true, pred, average="weighted", zero_division=0),
    }

individual_results = [
    metrics_row("Logistic Regression", y_test_enc, lr_pred),
    metrics_row("Decision Tree", y_test_enc, dt_pred),
    metrics_row("Random Forest", y_test_enc, rf_pred),
    metrics_row("XGBoost", y_test_enc, xgb_pred),
]

pd.DataFrame(individual_results).sort_values("F1", ascending=False).reset_index(drop=True)


,Model,Accuracy,Precision,Recall,F1
0,XGBoost,0.650625,0.670952,0.650625,0.650905
1,Logistic Regression,0.648350,0.670450,0.648350,0.647281
2,Decision Tree,0.632400,0.649404,0.632400,0.632597
3,Random Forest,0.614450,0.619737,0.614450,0.607618


## Hard Voting

In [11]:
predictions = np.vstack([lr_pred, dt_pred, rf_pred, xgb_pred])

hard_vote_pred = np.apply_along_axis(
    lambda row: np.bincount(row, minlength=len(label_encoder.classes_)).argmax(),
    axis=0,
    arr=predictions,
)

hard_voting_results = metrics_row("Hard Voting", y_test_enc, hard_vote_pred)
hard_voting_results


{'Model': 'Hard Voting',
 'Accuracy': 0.648775,
 'Precision': 0.664086236363555,
 'Recall': 0.648775,
 'F1': 0.648855191898442}

## Soft Voting

In [ ]:
lr_class_indices = label_encoder.transform(lr_model.classes_)

lr_proba_encoded = np.zeros_like(lr_proba)

for i, class_idx in enumerate(lr_class_indices):
    lr_proba_encoded[:, class_idx] = lr_proba[:, i]

# Average probabilities from all four models
probabilities = (lr_proba_encoded +dt_proba +rf_proba +xgb_proba) / 4

soft_vote_pred = np.argmax(probabilities, axis=1)

soft_voting_results = metrics_row("Soft Voting",y_test_enc,soft_vote_pred)

soft_voting_results

{'Model': 'Soft Voting',
 'Accuracy': 0.644525,
 'Precision': 0.6606296306597244,
 'Recall': 0.644525,
 'F1': 0.644588850070323}

## Bagging

In [ ]:
bagging_preds = []

for i in range(5):
    rng = np.random.RandomState(RANDOM_STATE + i)
    indices = rng.choice(len(X_train_dense),size=len(X_train_dense),replace=True,)

    bag_model = DecisionTreeClassifier(max_depth=12,random_state=RANDOM_STATE + i,)

    bag_model.fit(X_train_dense[indices],y_train_enc[indices],)

    bagging_preds.append(bag_model.predict(X_test_dense))

bagging_pred = np.apply_along_axis(lambda row: np.bincount(row, minlength=len(label_encoder.classes_)).argmax(),axis=0,arr=np.array(bagging_preds),)

bagging_results = metrics_row("Bagging", y_test_enc, bagging_pred)
bagging_results


{'Model': 'Bagging',
 'Accuracy': 0.648525,
 'Precision': 0.6686912305510232,
 'Recall': 0.648525,
 'F1': 0.6486196629397962}

## Boosting

**XGBoost is the boosting implementation**, so it is not reported as a second duplicate model.


In [14]:
boosting_results = metrics_row("Boosting (XGBoost)", y_test_enc, xgb_pred)
boosting_results


{'Model': 'Boosting (XGBoost)',
 'Accuracy': 0.650625,
 'Precision': 0.6709523841473113,
 'Recall': 0.650625,
 'F1': 0.650904539222079}

## Final Comparison

In [ ]:
all_results = pd.DataFrame(individual_results+ [hard_voting_results,soft_voting_results,bagging_results])

all_results = (all_results.sort_values("F1", ascending=False).reset_index(drop=True))

all_results


,Model,Accuracy,Precision,Recall,F1
0,XGBoost,0.650625,0.670952,0.650625,0.650905
1,Hard Voting,0.648775,0.664086,0.648775,0.648855
2,Bagging,0.648525,0.668691,0.648525,0.648620
3,Logistic Regression,0.648350,0.670450,0.648350,0.647281
4,Soft Voting,0.644525,0.660630,0.644525,0.644589
5,Decision Tree,0.632400,0.649404,0.632400,0.632597
6,Random Forest,0.614450,0.619737,0.614450,0.607618


In [16]:
print("Label encoder:", label_encoder.classes_)
print("LR classes:", lr_model.classes_)
print("DT classes:", dt_model.classes_)
print("RF classes:", rf_model.classes_)
print("XGB classes:", xgb_model.classes_)

Label encoder: ['Critical' 'High' 'Low' 'Medium']
LR classes: ['Critical' 'High' 'Low' 'Medium']
DT classes: [0 1 2 3]
RF classes: [0 1 2 3]
XGB classes: [0 1 2 3]


### Interpretation

Use the final table to determine whether combining models improves over the strongest individual classifier.

For the report, compare **Accuracy, Precision, Recall and F1-score**, and explicitly state whether the best ensemble actually improves the final Logistic Regression baseline.
